# Hafta 14 - Otomatik Blog Yazarı

Bu defterde Gemini API ve prompt mühendisliği teknikleri kullanarak otomatik SEO uyumlu blog yazıları üreteceğiz.

## Öğrenme Hedefleri
- Prompt mühendisliği (prompt engineering) temelleri
- SEO uyumlu yapılandırılmış içerik üretimi
- Chain-of-Thought (düşünce zinciri) prompting
- Few-Shot (az örnekli) prompting
- HTML/Markdown formatında çıktı üretimi
- Toplu (batch) içerik üretimi

In [ ]:
!pip install -q google-generativeai

### Kütüphanelerin Yüklenmesi

Projede kullanacağımız kütüphaneleri içe aktarıyoruz:

| Kütüphane | Amacı |
|-----------|-------|
| `IPython` | Yardımcı kütüphane |
| `google` | Google Gemini AI API |
| `json` | JSON veri formatı işleme |
| `time` | Yardımcı kütüphane |


In [ ]:
import google.generativeai as genai
from IPython.display import Markdown, display, HTML
import json
import time

API_KEY = "YOUR_API_KEY"
genai.configure(api_key=API_KEY)

model = genai.GenerativeModel('gemini-2.5-flash')
print("Hazır!")

## 1. Temel Blog Yazısı Üretimi

Önce basit bir prompt ile blog yazısı üretelim, sonra adım adım geliştirelim.

In [ ]:
# Basit prompt (kötü örnek)
basit_prompt = "Yapay zeka hakkında bir blog yazısı yaz."

response = model.generate_content(basit_prompt)
display(Markdown("### Basit Prompt Sonucu"))
display(Markdown(response.text[:500] + "..."))

## 2. Yapılandırılmış Prompt ile SEO Uyumlu Blog

İyi bir blog yazısı prompt'u şunları içermelidir:
- Hedef kitle tanımı
- Yazı yapısı (başlık, alt başlıklar, paragraflar)
- SEO gereksinimleri (meta açıklama, anahtar kelimeler)
- Ton ve üslup
- Uzunluk

In [ ]:
def blog_yazisi_uret(konu, hedef_kitle="genel okuyucu", kelime_sayisi=800):
    """SEO uyumlu blog yazısı üret."""
    
    prompt = f"""Sen profesyonel bir Türkçe blog yazarısın. Aşağıdaki konuda SEO uyumlu bir blog yazısı yaz.

KONU: {konu}
HEDEF KİTLE: {hedef_kitle}
UZUNLUK: Yaklaşık {kelime_sayisi} kelime

YAZI YAPISI (bu sırayla):
1. **Başlık (H1)**: Dikkat çekici, SEO uyumlu, 60 karakterden kısa
2. **Meta Açıklama**: 155 karakterlik SEO meta description
3. **Giriş Paragrafı**: Okuyucuyu çeken, problemi tanımlayan giriş
4. **Ana Bölümler (H2)**: En az 3 alt başlık, her biri 2-3 paragraf
5. **Pratik İpuçları**: Madde madde uygulanabilir tavsiyeler
6. **Sonuç**: Özet ve harekete geçirici mesaj (CTA)
7. **Etiketler**: Virgülle ayrılmış 5-7 SEO etiketi

SEO KURALLARI:
- Ana anahtar kelimeyi başlıkta ve ilk paragrafta kullan
- Alt başlıklarda anahtar kelime varyasyonları kullan
- Kısa paragraflar (3-4 cümle)
- Geçiş cümleleri kullan
- Aktif cümleler tercih et

FORMAT: Markdown
"""
    
    response = model.generate_content(prompt)
    return response.text

# Blog yazısı üret
blog = blog_yazisi_uret(
    konu="Yapay Zeka ile Kişiselleştirilmiş Eğitim",
    hedef_kitle="öğretmenler ve eğitim yöneticileri",
    kelime_sayisi=600
)

display(Markdown(blog))

## 3. Chain-of-Thought (Düşünce Zinciri) Prompting

Model, karmaşık görevlerde adım adım düşünmeye yönlendirildiğinde daha iyi sonuçlar üretir.

```
Normal Prompt:       "Blog yazısı yaz"  →  [Sonuç]
Chain-of-Thought:    "Önce planla → Araştır → Taslak çıkar → Yaz → Düzenle"  →  [Daha iyi sonuç]
```

In [ ]:
def cot_blog_yazisi(konu):
    """Chain-of-Thought ile adım adım blog yazısı üret."""
    
    cot_prompt = f"""Aşağıdaki konu hakkında bir blog yazısı üreteceksin. Adım adım düşün:

KONU: {konu}

ADIM 1 - ANALİZ:
Bu konu hakkında düşün:
- Bu konunun ana mesajı ne?
- Hedef okuyucu kim?
- Hangi alt konuları kapsamalı?
- Hangi anahtar kelimeler önemli?

ADIM 2 - YAPI PLANI:
Yazının iskeletini oluştur:
- Başlık seçenekleri (3 adet)
- Alt başlıklar
- Her bölümün ana fikri

ADIM 3 - YAZIM:
Planı takip ederek yazıyı yaz.

ADIM 4 - SEO OPTİMİZASYONU:
- Meta açıklama ekle
- Etiketler belirle
- Anahtar kelime yoğunluğunu kontrol et

Her adımı açıkça göster, ardından son blog yazısını üret.
"""
    
    response = model.generate_content(cot_prompt)
    return response.text

# Chain-of-Thought ile blog üret
cot_blog = cot_blog_yazisi("Python ile Veri Görselleştirme")
display(Markdown(cot_blog))

## 4. Few-Shot Prompting (Az Örnekli Yönlendirme)

Modele birkaç örnek vererek istediğimiz format ve üslubu öğretebiliriz.

In [ ]:
def few_shot_blog_giris(konu):
    """Few-shot prompting ile blog girişi üret."""
    
    prompt = f"""Aşağıdaki örneklerdeki üslup ve yapıyı takip ederek yeni bir blog girişi yaz.

=== ÖRNEK 1 ===
Konu: Uzaktan Çalışma
Giriş: Sabah alarmı çaldığında artık trafik stresini düşünmenize gerek yok. Pijamalarınızla 
kahvenizi yudumlarken işe başlayabilirsiniz. Kulağa harika geliyor, değil mi? Ancak uzaktan 
çalışmanın görünmeyen zorlukları da var. Bu yazıda, evden çalışırken verimliliğinizi artırmanın 
kanıtlanmış 7 yolunu keşfedeceksiniz.

=== ÖRNEK 2 ===
Konu: Sağlıklı Beslenme
Giriş: "Ne yediysek oyuz" demiş atalarımız. Ama günümüzde ne yediğimizi gerçekten biliyor 
muyuz? Süpermarket raflarında yüzlerce ürün bize sağlıklı olduğunu iddia ederken, asıl sağlıklı 
beslenmenin sırrı şaşırtıcı derecede basit. Bu rehberde, bilimsel araştırmalara dayanan 
beslenme ipuçlarını paylaşacağız.

=== ÖRNEK 3 ===
Konu: Dijital Detoks
Giriş: Telefonunuzu son 10 dakikada kaç kez kontrol ettiniz? İstatistiklere göre ortalama bir 
insan günde 96 kez telefonuna bakıyor. Bu sayı sizi şaşırttıysa, bir dijital detoks zamanı 
gelmiş olabilir. İşte ekran bağımlılığından kurtulmanın ve hayatınızı geri kazanmanın yolları.

=== YENİ KONU ===
Konu: {konu}
Giriş:"""
    
    response = model.generate_content(prompt)
    return response.text

# Few-shot ile farklı konularda giriş üret
konular = [
    "Yapay Zeka ve İş Geleceği",
    "Çocuklara Kodlama Öğretmek",
    "Sürdürülebilir Teknoloji"
]

for konu in konular:
    print(f"\n{'='*50}")
    print(f"KONU: {konu}")
    print(f"{'='*50}")
    giris = few_shot_blog_giris(konu)
    print(giris)
    print()

## 5. HTML Formatında Blog Çıktısı

Blog yazısını doğrudan HTML formatında üretip görüntüleyebiliriz.

In [ ]:
def html_blog_uret(konu):
    """HTML formatında blog yazısı üret."""
    
    prompt = f"""Aşağıdaki konu hakkında bir blog yazısı yaz ve HTML formatında döndür.

KONU: {konu}

HTML KURALLARI:
- <article> etiketi ile sar
- Başlık: <h1>
- Alt başlıklar: <h2>
- Paragraflar: <p>
- Listeler: <ul><li> veya <ol><li>
- Önemli kelimeler: <strong>
- Alıntılar: <blockquote>
- Meta bilgileri: <meta> etiketleri (description ve keywords)

CSS stili ekleme, sadece HTML yapısı ver.
Sadece HTML kodunu döndür, açıklama ekleme.
"""
    
    response = model.generate_content(prompt)
    # ```html ... ``` temizliği
    html_text = response.text.strip()
    if html_text.startswith("```"):
        html_text = html_text.split("\n", 1)[1]
        html_text = html_text.rsplit("```", 1)[0]
    return html_text

html_icerik = html_blog_uret("Veri Bilimi ile Kariyer Fırsatları")
display(HTML(html_icerik))

## 6. Toplu Blog Üretimi (Batch Generation)

Birden fazla konuda blog yazısı üretelim.

In [ ]:
def toplu_blog_uret(konu_listesi):
    """Birden fazla konu için blog özetleri üret."""
    sonuclar = []
    
    for i, konu in enumerate(konu_listesi, 1):
        print(f"[{i}/{len(konu_listesi)}] Üretiliyor: {konu}...")
        
        prompt = f"""Aşağıdaki konu için blog yazısı taslağı üret. JSON formatında döndür:

Konu: {konu}

JSON formatı:
{{
    "baslik": "SEO uyumlu başlık",
    "meta_aciklama": "155 karakterlik meta açıklama",
    "alt_basliklar": ["Alt başlık 1", "Alt başlık 2", "Alt başlık 3"],
    "anahtar_kelimeler": ["kelime1", "kelime2", "kelime3"],
    "hedef_kitle": "Hedef kitle açıklaması",
    "tahmini_okuma_suresi": "X dakika",
    "giris_paragrafi": "İlk paragraf"
}}

Sadece JSON döndür.
"""
        
        try:
            response = model.generate_content(prompt)
            json_text = response.text.strip()
            if json_text.startswith("```"):
                json_text = json_text.split("\n", 1)[1]
                json_text = json_text.rsplit("```", 1)[0]
            
            veri = json.loads(json_text)
            veri['konu'] = konu
            sonuclar.append(veri)
            print(f"  Başlık: {veri['baslik']}")
        except Exception as e:
            print(f"  Hata: {e}")
            sonuclar.append({'konu': konu, 'hata': str(e)})
        
        time.sleep(1)  # API rate limit için kısa bekleme
    
    return sonuclar

# Toplu üretim
konular = [
    "Yapay Zeka ile Tarımda Devrim",
    "Siber Güvenlik Temelleri",
    "Bulut Bilişim Nedir?",
    "Blokzincir Teknolojisi ve Geleceği"
]

blog_taslaklari = toplu_blog_uret(konular)

### Sonuçları düzenli göster

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# Sonuçları düzenli göster
for i, taslak in enumerate(blog_taslaklari, 1):
    if 'hata' in taslak:
        print(f"\n{i}. {taslak['konu']} - HATA: {taslak['hata']}")
        continue
    
    print(f"\n{'='*60}")
    print(f"{i}. {taslak.get('baslik', 'Başlık yok')}")
    print(f"{'='*60}")
    print(f"Meta: {taslak.get('meta_aciklama', '-')}")
    print(f"Hedef Kitle: {taslak.get('hedef_kitle', '-')}")
    print(f"Okuma Süresi: {taslak.get('tahmini_okuma_suresi', '-')}")
    print(f"Anahtar Kelimeler: {', '.join(taslak.get('anahtar_kelimeler', []))}")
    print(f"\nAlt Başlıklar:")
    for ab in taslak.get('alt_basliklar', []):
        print(f"  - {ab}")
    print(f"\nGiriş: {taslak.get('giris_paragrafi', '-')[:150]}...")

## 7. Prompt Mühendisliği İpuçları

### İyi Prompt Yazma Kuralları

| Kural | Kötü Örnek | İyi Örnek |
|-------|-----------|----------|
| **Spesifik ol** | "Bir şey yaz" | "500 kelimelik SEO uyumlu blog yazısı yaz" |
| **Rol ver** | "Blog yaz" | "Sen deneyimli bir teknoloji editörüsün..." |
| **Format belirt** | "Bilgi ver" | "Markdown formatında, madde madde listele" |
| **Kısıtlama ekle** | "Anlat" | "3 paragrafta, basit dille, örneklerle" |
| **Örnek ver** | "Böyle yaz" | "Şu örneği takip et: [örnek]" |
| **Adım adım iste** | "Yap" | "Önce analiz et, sonra planla, sonra yaz" |

### Prompt Şablonu

```
[ROL]: Sen bir ... uzmanısın.
[GÖREV]: ... konusunda ... yaz.
[FORMAT]: ... formatında döndür.
[KISITLAMALAR]: ... kurallarına uy.
[ÖRNEKLER]: İşte bir örnek: ...
[ÇIKTI]: Sadece ... döndür.
```

## Özet

| Teknik | Açıklama | Kullanım Alanı |
|--------|----------|----------------|
| **Yapılandırılmış Prompt** | Detaylı talimatlar ve format belirtme | SEO blog yazıları |
| **Chain-of-Thought** | Adım adım düşünme yönlendirmesi | Karmaşık içerik üretimi |
| **Few-Shot** | Örneklerle üslup öğretme | Tutarlı ton ve format |
| **JSON Çıktı** | Yapılandırılmış veri üretimi | Otomatik işleme |
| **Toplu Üretim** | Birden fazla içerik otomasyonu | İçerik pazarlaması |

### Alıştırma
1. Kendi blog konunuzu seçin ve `blog_yazisi_uret()` fonksiyonunu kullanarak tam bir yazı üretin.
2. Few-shot prompting ile kendi yazı üslubunuzu modele öğretin.
3. 5 farklı konu için toplu blog taslağı üretin ve karşılaştırın.